# 🚀 Ollama Connection Test

This notebook verifies that the Jupyter container can reach your local Ollama instance.

**Prerequisites:**
- Ollama is running on your host machine (`ollama serve`)
- At least one model is pulled (e.g. `ollama pull llama3.1`)

In [8]:
import os
import requests
from dotenv import load_dotenv

# Load config from .env
load_dotenv("/workspace/.env", override=True)

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://host.docker.internal:11434")
OLLAMA_DEFAULT_MODEL = os.getenv("OLLAMA_DEFAULT_MODEL", "llama3.1")
OPENAI_SDK_BASE_URL = os.getenv("OPENAI_SDK_BASE_URL", "http://host.docker.internal:11434/v1")
OLLAMA_FAKE_API_KEY = os.getenv("OLLAMA_FAKE_API_KEY", "ollama")

# Auto-detect: if the configured model isn't available, use the first one found
try:
    _resp = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
    _models = [m['name'] for m in _resp.json().get('models', [])]
    if OLLAMA_DEFAULT_MODEL not in _models and _models:
        # Try matching without tag suffix (e.g. 'llama3.1' matches 'llama3.1:latest')
        _match = [m for m in _models if m.startswith(OLLAMA_DEFAULT_MODEL)]
        if _match:
            OLLAMA_DEFAULT_MODEL = _match[0]
        else:
            print(f"⚠️  Model '{OLLAMA_DEFAULT_MODEL}' not found. Using '{_models[0]}' instead.")
            OLLAMA_DEFAULT_MODEL = _models[0]
except Exception:
    pass  # Will fail in the next cell with a clearer message

print(f"Ollama URL:        {OLLAMA_BASE_URL}")
print(f"OpenAI SDK URL:    {OPENAI_SDK_BASE_URL}")
print(f"Default model:     {OLLAMA_DEFAULT_MODEL}")

Ollama URL:        http://host.docker.internal:11434
OpenAI SDK URL:    http://host.docker.internal:11434/v1
Default model:     llama3.1:latest


## 1. Check Ollama is reachable

In [9]:
try:
    resp = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
    resp.raise_for_status()
    models = resp.json().get("models", [])
    print(f"✅ Ollama is reachable! Found {len(models)} model(s):")
    for m in models:
        size_gb = m.get('size', 0) / 1e9
        print(f"   • {m['name']} ({size_gb:.1f} GB)")
except Exception as e:
    print(f"❌ Cannot reach Ollama at {OLLAMA_BASE_URL}")
    print(f"   Error: {e}")
    print(f"   Make sure Ollama is running on your host machine.")

✅ Ollama is reachable! Found 1 model(s):
   • llama3.1:latest (4.9 GB)


## 2. Test with the `ollama` Python client

In [10]:
from ollama import Client

client = Client(host=OLLAMA_BASE_URL)

response = client.chat(
    model=OLLAMA_DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)

print(f"Model: {OLLAMA_DEFAULT_MODEL}")
print(f"Response: {response['message']['content']}")

Model: llama3.1:latest
Response: Hello, how can I assist you today?


## 3. Test with OpenAI SDK (Ollama-compatible endpoint)

Ollama exposes an OpenAI-compatible API at `/v1`. This means you can use the standard OpenAI Python SDK to talk to your local models — useful for code that's designed to work with OpenAI but you want to run locally.

In [11]:
from openai import OpenAI

openaisdk_client = OpenAI(
    base_url=OPENAI_SDK_BASE_URL,
    api_key=OLLAMA_FAKE_API_KEY,  # fake key — Ollama doesn't require auth
)

response = openaisdk_client.chat.completions.create(
    model=OLLAMA_DEFAULT_MODEL,
    messages=[{"role": "user", "content": "What is 2 + 2? Answer in one word."}],
    temperature=0.0,
)

print(f"Model: {response.model}")
print(f"Response: {response.choices[0].message.content}")

Model: llama3.1:latest
Response: Four.


## 4. Test with LangChain + Ollama

In [12]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=OLLAMA_DEFAULT_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0.7,
)

result = llm.invoke("What is RAG in the context of LLMs? Answer in 2 sentences.")
print(result.content)

In the context of Large Language Models (LLMs), RAG stands for "Reformer-based Architecture" or more broadly, "Retrieval-Augmented Generation", which refers to a class of models that use a combination of retrieval and generation mechanisms to produce text outputs. Specifically, RAG involves training an LLM to retrieve relevant information from a knowledge base or database, and then generating text based on this retrieved information using a generator model.


---
✅ **All checks passed!** You're ready to start building with Ollama in this environment.